# Automated Topology & Dynamic Simulation Toolkit

### Industrialized Workflow
1. **Environment & Parsing:** Set up Dynawo and extract network topology directly from Modelica files.
2. **Construct Network:** Automatically build the PyPowSyBl static structure from the parsed Modelica data.
3. **Visualize & Loadflow:** Render SVG network diagrams and execute the base AC Loadflow.
4. **Cross-Validation:** Benchmark the loadflow voltage and angle results against OpenModelica ground truth.
5. **Model Compilation (optional):** Compile the Dynawo models of the case, instead of using the models precompiled in the Dynawo model database.
6. **Dynamic Linking & Parameters:** Map Modelica equations to static components and generate `.PAR` physical parameters.
7. **Execute Dynamic Simulation:** Run time-domain simulations and plot transient responses.
8. **Modelica Initialization:** Extract dump values via CLI fallback to automatically generate a fully initialized Modelica model.
9. **Initialization Check:** Simulate the initialized model in OpenModelica and compare its initial point with the loadflow.

In [ ]:
# --- 1. ENVIRONMENT SETUP ---
import os
import re
from pathlib import Path


def setup_dynawo_environment():
    """Finds and configures the Dynawo installation path."""
    config_dir = os.path.expanduser("~/.itools")
    config_file = os.path.join(config_dir, "config.yml")
    dynawo_path = None

    if os.path.exists(config_file):
        with open(config_file, "r") as file:
            match = re.search(r'homeDir:\s*["\']?([^"\'\n]+)["\']?', file.read())
            if match:
                dynawo_path = match.group(1)

    if not dynawo_path or not os.path.exists(dynawo_path):
        raise FileNotFoundError("Error! A valid Dynawo installation was not found.")

    os.environ["DYNAWO_SYS_PATH"] = dynawo_path
    return dynawo_path, config_dir


def configure_powsybl_template(dynawo_path, config_dir):
    """Overwrites the PyPowSyBl configuration with the local template."""
    current_dir = os.getcwd()
    target_config = os.path.join(config_dir, "config.yml")
    template_config = "./om_powsybl_data/config.yml"

    if os.path.exists(template_config):
        with open(template_config, "r") as file:
            template_content = file.read()

        template_content = template_content.replace('"WORKING_DIR/dynawo"', f'"{dynawo_path}"')
        template_content = template_content.replace("WORKING_DIR", current_dir)

        os.makedirs(config_dir, exist_ok=True)
        with open(target_config, "w") as file:
            file.write(template_content)


# Execute environment setup
dynawo_sys_path, global_config_dir = setup_dynawo_environment()
configure_powsybl_template(dynawo_sys_path, global_config_dir)
print("Environment successfully configured!")

In [ ]:
# --- 2. IMPORT LIBRARIES & DEFINE GLOBAL CONFIGURATION ---
# Core Toolkit Imports
from python_pypowsybl.Code.core.visualizer import NetworkVisualizer
from python_pypowsybl.Code.core.mo_topology import MoTopologyToolkit
from python_pypowsybl.Code.core.powerflow import PowerFlowRunner
from python_pypowsybl.Code.core.comparison import LoadFlowComparator
from python_pypowsybl.Code.core.model_compiler import DynawoModelCompiler
from python_pypowsybl.Code.core.model_linker import link_models
from python_pypowsybl.Code.core.parameter_generator import DynamicParameterGenerator
from python_pypowsybl.Code.core.simulation import DynawoSimulationRunner
from python_pypowsybl.Code.core.init_generator import ModelicaInitGenerator

# Additional Libraries
import pypowsybl.dynamic as dyn
from IPython.display import SVG, display, HTML
import matplotlib.pyplot as plt

# Base Network Configurations
SOURCE_DIRECTORY = "./om_powsybl_data/SimpleCases/"
MODEL_NAME = "BESS"
DYNAWO_PACKAGE_PATH = os.environ["DYNAWO_SYS_PATH"] + "/ddb/Dynawo/package.mo"
LOCAL_MODELICA_FILES = ["BESS.mo"]
NETWORK_NAME = "BESS"
EXPORT_DIRECTORY = f"grid_export_{NETWORK_NAME}"

# Dynawo models of the dynamic simulation (step 8): None uses the models precompiled in
# the Dynawo model database, "library" compiles the Dynawo models of the case from the
# Dynawo Modelica library, and "case" compiles the Modelica classes of the case, user
# classes included
MODEL_COMPILATION = None

# Ensure export directory exists
os.makedirs(EXPORT_DIRECTORY, exist_ok=True)
print("Libraries imported and configuration variables defined successfully.")

In [ ]:
# --- 3. INSTANTIATE TOOLKIT & PARSE ELECTRICAL DATA ---
topology_toolkit = MoTopologyToolkit(
    SOURCE_DIRECTORY, MODEL_NAME, DYNAWO_PACKAGE_PATH, LOCAL_MODELICA_FILES
)

# Parse the electrical data from the Modelica files into a structured representation
parsed_electrical_data = topology_toolkit.parse_electrical_data()

# Export to JSON for structural verification
json_export_filepath = os.path.join(EXPORT_DIRECTORY, f"grid_export_{NETWORK_NAME}.json")
topology_toolkit.export_to_standard_json(parsed_electrical_data, json_export_filepath)

# Re-import to ensure data integrity
parsed_electrical_data = topology_toolkit.import_from_standard_json(json_export_filepath)

generator_count = len(parsed_electrical_data.get("generators", []))
print(f"Parsing successful. Found {generator_count} generators.")

# Initial parameters without value: the initialization computes them in the last steps
missing_parameters = {
    name: info["missing_initial_parameters"]
    for category in parsed_electrical_data.values()
    for name, info in category.items()
    if info.get("missing_initial_parameters")
}
if missing_parameters:
    print("\nInitial parameters left to the initialization:")
    for name, parameters in missing_parameters.items():
        print(f"- {name}: {', '.join(parameters)}")

In [ ]:
# --- 4. CONSTRUCT POWSYBL NETWORK ---
parsed_electrical_data["source_xiidm"] = Path("om_powsybl_data/Base_Case.xiidm").resolve()

# The slack bus is the infinite bus or the source without active power set point of the
# Modelica model. If there is none, it is determined by pattern matching
# (slack_bus_mapping.json), and otherwise the largest participating generator is used.
power_network = topology_toolkit.build_powsybl_network(parsed_electrical_data)

print("Power Network constructed successfully. Summary:")
print(f"- Buses: {len(power_network.get_buses())}")
print(f"- Lines: {len(power_network.get_lines())}")
print(f"- Generators: {len(power_network.get_generators())}")
print(f"- Loads: {len(power_network.get_loads())}")
print(f"- Shunts: {len(power_network.get_shunt_compensators())}")
print(f"- Transformers: {len(power_network.get_2_windings_transformers())}")

In [ ]:
# --- 5. NETWORK VISUALIZATION ---
def render_network_visualizations(network):
    """Generates and displays SVG diagrams of the network structure."""
    print("Rendering Network Diagrams...")
    network_diagrams = NetworkVisualizer.generate_full_system_diagrams(network)

    # 1. Display Macro View (Overview of Lines & Substations)
    if "network_area" in network_diagrams:
        display(HTML("<h3>Macro View: Overall Network Area</h3>"))
        display(SVG(str(network_diagrams["network_area"])))

    # 2. Display Micro Views (Detailed Substations)
    display(HTML("<h3>Micro View: Detailed Substations (Generators visible)</h3>"))
    for diagram_name, svg_content in network_diagrams.items():
        if diagram_name != "network_area":
            display(HTML(f"<b>Substation: {diagram_name}</b>"))
            display(SVG(str(svg_content)))


# Execute visualization
render_network_visualizations(power_network)

In [ ]:
# --- 6. AC LOAD FLOW ANALYSIS ---
def run_and_display_loadflow(network, export_path):
    """Executes AC Load Flow analysis and displays structured results."""
    print("Starting AC Load Flow analysis...")

    # Execute the Load Flow (logs will save to the export folder if divergence occurs)
    is_converged = PowerFlowRunner.run_ac_loadflow(network, export_path=export_path)

    if not is_converged:
        print("\nFAILED: Load flow did NOT converge. Check the export folder for logs.")
        return

    print("\nSUCCESS: Load flow CONVERGED!")

    # Extract and display Bus Data (Voltages and Angles)
    print("\n--- BUS RESULTS (Voltages & Angles) ---")
    try:
        buses_dataframe = network.get_buses()
        display(buses_dataframe[["v_mag", "v_angle"]])
    except Exception as error:
        print(f"Could not retrieve Bus results: {error}")

    # Extract and display Generator Data (Active & Reactive Power)
    print("\n--- GENERATOR RESULTS (Active & Reactive Power) ---")
    try:
        generators_dataframe = network.get_generators()
        display(generators_dataframe[["bus_id", "p", "q", "target_p", "target_v"]])
    except Exception as error:
        print(f"Could not retrieve Generator results: {error}")

    # Extract and display Line Data
    print("\n--- LINE RESULTS (Power Flows & Currents) ---")
    try:
        lines_dataframe = network.get_lines()
        display(lines_dataframe[["bus1_id", "bus2_id", "p1", "q1", "p2", "q2", "i1", "i2"]])
    except Exception as error:
        print(f"Could not retrieve Line results: {error}")


# Execute load flow and save the network state
run_and_display_loadflow(power_network, EXPORT_DIRECTORY)
topology_toolkit.save_powsybl_network(power_network, export_path=EXPORT_DIRECTORY)

In [ ]:
# --- 7. CROSS-VALIDATION WITH OPENMODELICA ---
def display_voltage_comparison(comparison_dataframe):
    """Displays a voltage comparison, highlighting the significant discrepancies."""

    # Style function to highlight errors larger than 0.001 pu (0.1%)
    def highlight_significant_errors(value):
        color = "orange" if isinstance(value, (int, float)) and value > 0.001 else "black"
        return f"color: {color}"

    styled_comparison = comparison_dataframe.style.map(
        highlight_significant_errors, subset=["Δ V (pu)", "Δ Theta (deg)"]
    ).format("{:.4f}", na_rep="N/A")

    display(styled_comparison)

    # Provide a summary of the metrics
    print("\n--- Benchmark Summary ---")
    print(f"Average Voltage Mismatch: {comparison_dataframe['Δ V (pu)'].mean():.6f} pu")
    print(f"Maximum Voltage Mismatch: {comparison_dataframe['Δ V (pu)'].max():.6f} pu")
    print(f"Average Angle Mismatch:   {comparison_dataframe['Δ Theta (deg)'].mean():.6f} deg")
    print(f"Maximum Angle Mismatch:   {comparison_dataframe['Δ Theta (deg)'].max():.6f} deg")


def validate_against_ground_truth(network, parsed_data):
    """
    Simulates the OpenModelica reference of the case at t = 0 and compares its voltages
    with the load flow. The reference is the initialized version of the model provided
    next to it (<Model>_initialized.mo), or else the model itself when it is already
    initialized.
    """
    print("Running Ground Truth Validation via OpenModelica...")

    comparison_dataframe, reference_model = LoadFlowComparator.compare_with_reference(
        network=network,
        parsed_data=parsed_data,
        dynawo_package_path=DYNAWO_PACKAGE_PATH,
        source_dir=SOURCE_DIRECTORY,
        local_files=LOCAL_MODELICA_FILES,
        model_name=MODEL_NAME,
    )

    if comparison_dataframe.empty:
        print(
            f"No OpenModelica reference could be simulated: {reference_model} is not initialized "
            "yet. Its initialized version is validated in the last step of the notebook."
        )
        return

    print(
        f"\nSUCCESS: Validation against {reference_model} completed! Displaying top discrepancies:"
    )
    display_voltage_comparison(comparison_dataframe)


# Execute validation
validate_against_ground_truth(power_network, parsed_electrical_data)

In [ ]:
# --- 8. DYNAWO MODEL COMPILATION (OPTIONAL) ---
# The Dynawo models of the components are compiled before the dynamic linking, with the
# model generator of the Dynawo distribution. It requires a distribution with on-the-fly
# model compilation, built with a compiler compatible with the C++ compiler of the system
# (see the README of the toolkit).
compiled_models = None
if MODEL_COMPILATION:
    print(f"Compiling the Dynawo models of the case (mode '{MODEL_COMPILATION}')...")
    model_compiler = DynawoModelCompiler(
        dynawo_sys_path, os.path.join(EXPORT_DIRECTORY, "compiled_models")
    )
    compiled_models = model_compiler.compile_case(
        parsed_electrical_data,
        mode=MODEL_COMPILATION,
        connector=topology_toolkit.connector,
        source_dir=SOURCE_DIRECTORY,
        local_files=LOCAL_MODELICA_FILES,
    )
    print(compiled_models.summary())
    display(compiled_models.to_dataframe())
else:
    print("Model compilation disabled: the models of the Dynawo model database are used.")

In [ ]:
# --- 9. DYNAMIC MODEL LINKING ---
print("Resolving Modelica equations to IIDM static components...\n")

# Every component whose Modelica class is in the model registry (dynamic_models.json) is
# mapped to the corresponding model of the Dynawo model database, or to the model
# compiled for it in step 8
dynamic_setup = link_models(power_network, parsed_electrical_data, compiled_models=compiled_models)

print("=== DYNAMICALLY LINKED MODELS REGISTRY ===")
for component_category, registry_dataframe in dynamic_setup.registry.items():
    print(f"\n--- {component_category.upper()} ---")
    # Display relevant routing columns cleanly
    print(registry_dataframe.to_string(index=False))

if dynamic_setup.full_dynamic_network:
    removed_generators = ", ".join(dynamic_setup.removed_elements)
    print(
        "\nThe case contains an infinite bus: every bus, line and transformer is simulated "
        "with its own dynamic model, and the generators standing for the infinite buses in "
        f"the load flow are removed from the simulated network ({removed_generators})."
    )

In [ ]:
# --- 10. DYNAMIC PHYSICAL PARAMETERS GENERATION ---
print("Extracting physical parameters from Dynawo DDB for dynamic simulation...")

# The parameter files are written to the export directory of the case: the model
# parameters take their initial values from the load flow and their other values from
# the Modelica model, and the solver parameters bound the time step.
parameter_files = DynamicParameterGenerator.generate_parameters(
    setup=dynamic_setup,
    parsed_data=parsed_electrical_data,
    dynawo_path=dynawo_sys_path,
    target_dir=EXPORT_DIRECTORY,
    solver="IDA",
    max_time_step=0.01,
)

print(f"Successfully generated dynamic physical parameters inside '{EXPORT_DIRECTORY}':")
for name, value in parameter_files.provider_parameters().items():
    print(f"- {name}: {value}")

### Initialization Workaround

The command line replay of step 12 acts as a temporary workaround. Once the `dumpInit` option is fully integrated natively through `dyn.Parameters`, it can be omitted.

**Current Workflow for Model Initialization:**
1. **Simulation through PyPowSyBl:** PyPowSyBl writes the Dynawo input files (DYD, PAR, jobs) from the dynamic model mapping and runs the simulation.
2. **File Retrieval:** The input files of that run are copied from the system's temporary directory to the export directory.
3. **CLI Re-execution:** The simulation is replayed with the Dynawo command line, enforcing `dumpInitValues` and applying the compatibility fixes required by the installed Dynawo version.

The curves of the replay are the results displayed below, and its dump holds the exact initialization state needed for Modelica.

In [ ]:
# --- 11. CONFIGURE DYNAMIC EVENTS & OBSERVERS ---
SIMULATION_STOP_TIME = 10.0

# Curves plotted after the simulation: one subplot per group, {curve column: label}
CURVE_PLOTS = [
    {
        "title": "BESS active and reactive power",
        "ylabel": "Power (pu, base SNom)",
        "curves": {
            "BESS_BESScb_measurements_PPu": "Active power",
            "BESS_BESScb_measurements_QPu": "Reactive power",
        },
    },
    {
        "title": "BESS terminal voltage",
        "ylabel": "Voltage (pu)",
        "curves": {"BESS_BESScb_measurements_UPu": "Terminal voltage"},
    },
]


def configure_simulation_events():
    """Maps topological events and sets up observers (curves)."""
    print("Configuring topological events and output observers...")

    # The events of this case belong to the Modelica model itself: the infinite bus with
    # variations applies a voltage dip to 0.55 pu between 1 s and 1.5 s, and a frequency
    # step to 1.01 pu between 6 s and 6.5 s. No additional event is needed.
    event_mapping = dyn.EventMapping()

    output_mapping = dyn.OutputVariableMapping()
    # Track the active power, reactive power and voltage measured at the BESS terminal
    output_mapping.add_dynamic_model_curves(
        "BESS", ["BESScb_measurements_PPu", "BESScb_measurements_QPu", "BESScb_measurements_UPu"]
    )

    print("Events and Observers mapped successfully.")
    return event_mapping, output_mapping


# Execute configuration
sim_events, sim_outputs = configure_simulation_events()

In [ ]:
# --- 12. EXECUTE TIME-DOMAIN SIMULATION ---
def run_dynamic_simulation(setup, files, events, outputs):
    """Runs the dynamic simulation and its command line replay, and plots the curves."""
    print("Running time-domain simulation via Dynawo engine...")
    simulation_runner = DynawoSimulationRunner(dynawo_sys_path, EXPORT_DIRECTORY)
    result = simulation_runner.run(
        setup, files, events, outputs, start_time=0.0, stop_time=SIMULATION_STOP_TIME
    )
    print(result.summary())

    if not result.succeeded:
        print("CRITICAL ERROR: Simulation failed. Check the Dynawo logs in the export folder.")
        return result

    print("SUCCESS: Time-domain simulation completed.")
    plot_transient_response(result.curves)
    return result


def plot_transient_response(curves_dataframe):
    """Plots the configured curves, one subplot per group."""
    figure, axes = plt.subplots(
        len(CURVE_PLOTS), 1, figsize=(10, 4 * len(CURVE_PLOTS)), sharex=True
    )
    axes = axes if len(CURVE_PLOTS) > 1 else [axes]

    for axis, plot in zip(axes, CURVE_PLOTS):
        for column, label in plot["curves"].items():
            axis.plot(curves_dataframe.index, curves_dataframe[column], label=label)
        axis.set_title(plot["title"])
        axis.set_ylabel(plot["ylabel"])
        axis.grid(True)
        axis.legend()

    axes[-1].set_xlabel("Time (Seconds)")
    figure.suptitle(f"Transient Response: {NETWORK_NAME}")
    figure.tight_layout()
    plt.show()


# Execute simulation
simulation_result = run_dynamic_simulation(dynamic_setup, parameter_files, sim_events, sim_outputs)

In [ ]:
# --- 13. GENERATE INITIALIZED MODELICA FILE ---
print("Extracting dump values and building the fully initialized Modelica file...")

initialized_model = ModelicaInitGenerator.generate_initialized_model(
    export_folder=EXPORT_DIRECTORY,
    source_dir=SOURCE_DIRECTORY,
    model_name=MODEL_NAME,
    setup=dynamic_setup,
    parsed_data=parsed_electrical_data,
    connector=topology_toolkit.connector,
    dynawo_path=dynawo_sys_path,
    dump_dir=simulation_result.init_values_directory,
)

if initialized_model:
    print("\nInitialization sequence completed successfully!")
    print(f"Initialized Modelica file saved at: {initialized_model.path}")
    print(f"Initialized model: {initialized_model.class_name}")
    print(f"\nInitial values set for {len(initialized_model.values)} components, for instance:")
    for component, values in list(initialized_model.values.items())[:5]:
        print(f"- {component}: {', '.join(values)}")
else:
    print("\nInitialization failed. Please check the export folder logs for details.")

In [ ]:
# --- 14. VALIDATE THE INITIALIZED MODELICA MODEL ---
# The initialized model is simulated in OpenModelica at t = 0 and its voltages are compared
# with the load flow: an initialized model starts exactly from that steady state.
print("Simulating the initialized model in OpenModelica...")

initialized_comparison = LoadFlowComparator.validate_initialized_model(
    network=power_network,
    parsed_data=parsed_electrical_data,
    initialized_model=initialized_model,
    dynawo_package_path=DYNAWO_PACKAGE_PATH,
    source_dir=SOURCE_DIRECTORY,
    local_files=LOCAL_MODELICA_FILES,
)

if initialized_comparison.empty:
    print("Validation failed: the initialized model could not be simulated in OpenModelica.")
else:
    print(f"\nSUCCESS: {initialized_model.class_name} validated! Displaying top discrepancies:")
    display_voltage_comparison(initialized_comparison)